# NB05 — Advanced Fine-Tuning (5-class, robustness-regularized)

Single-task 5-class fine-tuning of BanglishBERT / MuRIL / XLM-R with the framework in plans.md §5:
**FGM adversarial training · R-Drop · EMA weights · multi-sample dropout · class-balanced focal ·
balanced sampler**, script-aware (BanglaBERT ← Bangla script only). Uniform LR (no decay).

Trains on the 70% split, early-stops on the 10% val, **reports on the pure 20% test**. Saves per-model
val+test logits for the ensemble (NB06). Every technique is a config toggle → feeds the NB08 ablation.


In [1]:
# ── Install (uncomment if env differs) ──
!pip install torch==2.4.1 transformers==4.44.2 scikit-learn==1.5.1 pandas==2.2.2 numpy==1.26.4 sentencepiece==0.2.0 --quiet
print("deps assumed present")

deps assumed present


ERROR: Could not find a version that satisfies the requirement torch==2.4.1 (from versions: 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0)

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for torch==2.4.1


In [2]:
import os, json, time, math, random, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import (f1_score, accuracy_score, matthews_corrcoef, roc_auc_score)
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA: True | GPU: NVIDIA GeForce RTX 4060 Ti


In [3]:
# ═══════════════════════════════════════════════════════════════════
# CONFIG — advanced 5-class. Every technique is a toggle (ablation-ready).
# ═══════════════════════════════════════════════════════════════════
DEBUG         = False        # smoke-test first; set False for the real run
DEBUG_SAMPLES = 400
DEBUG_EPOCHS  = 2

CONFIG = {
    "models": {
    "banglishbert": "csebuetnlp/banglishbert",   # replaces banglabert — handles both scripts
    "banglabert_l": "csebuetnlp/banglabert_large", # new 4th model — bangla-script expert, BS=8
    "muril":        "google/muril-base-cased",
    "xlmr":         "xlm-roberta-base",
    },
    "max_length": 128, "batch_size": 16, "eval_batch_size": 32, "grad_accum_steps": 2,
    "num_workers": 0, "epochs": 8, "encoder_lr": 2e-5, "head_lr": 8e-5,
    "weight_decay": 0.01, "warmup_ratio": 0.10, "max_grad_norm": 1.0,
    "label_smoothing": 0.03, "focal_gamma": 2.0, "class_weight_beta": 0.999,
    "dropout": 0.25, "head_hidden_dim": 384, "n_msd": 4, "patience": 3,
    "use_fp16": torch.cuda.is_available(),
    # ── advanced toggles ──
    "use_balanced_sampler": True, "sampler_alpha": 0.5,
    "per_class_loss_multiplier": {},
    "use_fgm":   True, "fgm_epsilon": 1.0,
    "use_rdrop": True, "rdrop_alpha": 1.0,
    "use_ema":   True, "ema_decay": 0.999,
    "banglabert_script_only": True,   # applies to banglabert_l only now
    "banglabert_script_key": "banglabert_l",  # tag to identify the script-only model
}
USE_LRDECAY = False   # constraint 7

LABEL_COL = "label5"
TASK = "label5"
TEXT_COL = "text_clean"
SPLIT_DIR = "../data/splits"
OUTPUT_DIR = "../outputs/models_main"
RUN_MODELS = ["banglishbert", "banglabert_l", "muril", "xlmr"]
RUN_SEEDS  = [42, 123, 456]
FORCE_RETRAIN = False
os.makedirs(OUTPUT_DIR, exist_ok=True)
if DEBUG:
    CONFIG["epochs"] = DEBUG_EPOCHS; RUN_SEEDS = [42]
    print("⚠ DEBUG mode")
print(f"toggles: sampler={CONFIG['use_balanced_sampler']} fgm={CONFIG['use_fgm']} "
      f"rdrop={CONFIG['use_rdrop']} ema={CONFIG['use_ema']} script_aware={CONFIG['banglabert_script_only']}")
print(f"runs: {len(RUN_MODELS)*len(RUN_SEEDS)} | LRdecay={USE_LRDECAY}")

toggles: sampler=True fgm=True rdrop=True ema=True script_aware=True
runs: 12 | LRdecay=False


In [4]:
# ── Load splits + build label encoder ───────────────────────────────────────
train_full = pd.read_csv(f"{SPLIT_DIR}/random_train.csv")
val_df     = pd.read_csv(f"{SPLIT_DIR}/random_val.csv")
test_df    = pd.read_csv(f"{SPLIT_DIR}/random_test.csv")
classes = sorted(train_full[LABEL_COL].unique())
label_enc = {c: i for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)
assert NUM_CLASSES == 5, f"expected 5 classes, got {NUM_CLASSES}"
print(f"classes ({NUM_CLASSES}): {label_enc}")
print(f"train={len(train_full):,} val={len(val_df):,} test={len(test_df):,} (test = pure 20%)")
if DEBUG:
    train_full = train_full.groupby(LABEL_COL, group_keys=False).apply(
        lambda g: g.sample(min(len(g), DEBUG_SAMPLES//NUM_CLASSES), random_state=42))
    val_df  = val_df.sample(min(len(val_df), 300), random_state=42)
    test_df = test_df.sample(min(len(test_df), 300), random_state=42)
with open(f"{OUTPUT_DIR}/label_encoder.json", "w") as f:
    json.dump(label_enc, f, indent=2)

classes (5): {'abusive': 0, 'none': 1, 'religious': 2, 'sexual': 3, 'threat': 4}
train=66,026 val=9,432 test=18,865 (test = pure 20%)


In [5]:
# ── Dataset / Collator ──────────────────────────────────────────────────────
class DS(Dataset):
    def __init__(self, df, tok, maxlen, enc):
        self.texts  = df.reset_index(drop=True)[TEXT_COL].fillna("").astype(str).tolist()
        self.labels = [int(enc.get(v, -1)) for v in df[LABEL_COL]]
        self.tok, self.maxlen = tok, maxlen
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        e = self.tok(self.texts[i], max_length=self.maxlen, truncation=True, padding=False)
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in e.items()}
        item["label"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

class Collator:
    def __init__(self, tok): self.tok = tok
    def __call__(self, fs):
        labels = torch.stack([f["label"] for f in fs])
        feats = [{k: v for k, v in f.items() if k != "label"} for f in fs]
        b = self.tok.pad(feats, padding=True, return_tensors="pt"); b["label"] = labels
        return b
print("dataset ok")

dataset ok


In [6]:
# ── Loss, class weights, sampler ────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__(); self.gamma, self.weight, self.ls = gamma, weight, label_smoothing
    def forward(self, logits, t):
        ce = F.cross_entropy(logits, t, weight=self.weight, reduction="none", label_smoothing=self.ls)
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

def build_class_weights(series, enc, beta=0.999, max_w=10.0, per_class_mult=None):
    mapped = series.map(enc).dropna().astype(int); n = len(enc)
    counts = mapped.value_counts().sort_index(); w = np.ones(n, dtype=np.float32)
    for i in range(n):
        c = int(counts.get(i, 0))
        if c > 0: w[i] = 1.0 / max((1.0 - beta**c) / max(1.0 - beta, 1e-12), 1e-9)
    w = np.clip(w, w.min(), w.min() * max_w)
    if per_class_mult:
        for cname, m in per_class_mult.items():
            if cname in enc: w[enc[cname]] *= float(m)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

def make_balanced_sampler(df, enc, alpha=0.5):
    y = df[LABEL_COL].map(enc).fillna(-1).astype(int).values
    counts = np.bincount(y[y >= 0], minlength=len(enc)).astype(float); counts[counts == 0] = 1.0
    cw = (1.0 / counts) ** float(alpha)
    sw = np.where(y >= 0, cw[np.clip(y, 0, None)], 0.0)
    return WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(sw), replacement=True)
print("loss/sampler ok")

loss/sampler ok


In [7]:
# ── Model: encoder + multi-sample-dropout head ──────────────────────────────
class MSDHead(nn.Module):
    def __init__(self, hidden, n_cls, dropout=0.25, inner=384, n_msd=4):
        super().__init__(); inner = min(inner, hidden)
        self.pre = nn.Sequential(nn.Linear(hidden, inner), nn.GELU(), nn.LayerNorm(inner))
        self.drops = nn.ModuleList([nn.Dropout(dropout) for _ in range(max(1, n_msd))])
        self.out = nn.Linear(inner, n_cls)
    def forward(self, x):
        h = self.pre(x)
        if self.training and len(self.drops) > 1:
            return torch.stack([self.out(d(h)) for d in self.drops], 0).mean(0)
        return self.out(self.drops[0](h))

class Encoder(nn.Module):
    def __init__(self, name, n_cls, dropout=0.25, inner=384, n_msd=4):
        super().__init__(); self.encoder = AutoModel.from_pretrained(name)
        h = self.encoder.config.hidden_size
        self._tti = self.encoder.config.model_type.lower() not in ("roberta", "xlm-roberta")
        self.head = MSDHead(h, n_cls, dropout, inner, n_msd)
    def _pool(self, out, mask):
        hs = out.last_hidden_state; cls = hs[:, 0]
        mean = (hs * mask.unsqueeze(-1).float()).sum(1) / mask.sum(1, keepdim=True).float().clamp(1)
        return 0.5 * cls + 0.5 * mean
    def forward(self, ids, mask, tti=None):
        kw = {"input_ids": ids, "attention_mask": mask}
        if self._tti and tti is not None: kw["token_type_ids"] = tti
        return self.head(self._pool(self.encoder(**kw), mask))

def uniform_params(model, enc_lr, head_lr, wd):
    nd = ["bias", "LayerNorm.weight", "LayerNorm.bias"]; g = []
    for grp, lr in [(model.encoder, enc_lr), (model.head, head_lr)]:
        dec, ndec = [], []
        for n, p in grp.named_parameters():
            if p.requires_grad: (ndec if any(x in n for x in nd) else dec).append(p)
        g += [{"params": dec, "lr": lr, "weight_decay": wd},
              {"params": ndec, "lr": lr, "weight_decay": 0.0}]
    return g
print("model ok")

model ok


In [8]:
# ── FGM adversarial + EMA + R-Drop KL ───────────────────────────────────────
class FGM:
    """Perturb token embeddings along the (normalized) gradient direction."""
    def __init__(self, model, epsilon=1.0, emb_key="word_embeddings"):
        self.model, self.eps, self.key, self.backup = model, epsilon, emb_key, {}
    def attack(self):
        for name, p in self.model.named_parameters():
            if p.requires_grad and self.key in name and p.grad is not None:
                self.backup[name] = p.data.clone()
                norm = torch.norm(p.grad)
                if norm != 0 and not torch.isnan(norm):
                    p.data.add_(self.eps * p.grad / norm)
    def restore(self):
        for name, p in self.model.named_parameters():
            if name in self.backup: p.data = self.backup[name]
        self.backup = {}

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
    def apply_shadow(self, model):
        self.backup = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad: p.data.copy_(self.shadow[n])
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.backup: p.data.copy_(self.backup[n])
        self.backup = {}

def rdrop_kl(lp, lq):
    p_log, q_log = F.log_softmax(lp, -1), F.log_softmax(lq, -1)
    p, q = p_log.exp(), q_log.exp()
    return 0.5 * ((p * (p_log - q_log)).sum(-1) + (q * (q_log - p_log)).sum(-1)).mean()
print("fgm/ema/rdrop ok")

fgm/ema/rdrop ok


In [9]:
# ── Metrics + logits ────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y, PR = [], [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        lg = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
        pr = F.softmax(lg, -1).cpu().numpy(); y = b["label"].cpu().numpy(); m = y >= 0
        P.extend(pr[m].argmax(-1)); Y.extend(y[m]); PR.extend(pr[m])
    y, p, pr = np.array(Y), np.array(P), np.array(PR)
    rec = {"macro_f1": round(float(f1_score(y, p, average="macro", zero_division=0)), 4),
           "weighted_f1": round(float(f1_score(y, p, average="weighted", zero_division=0)), 4),
           "accuracy": round(float(accuracy_score(y, p)), 4),
           "mcc": round(float(matthews_corrcoef(y, p)), 4)}
    try:
        rec["auroc"] = round(float(roc_auc_score(y, pr, multi_class="ovr", average="macro",
                                                 labels=list(range(pr.shape[1])))), 4)
    except Exception:
        rec["auroc"] = float("nan")
    return rec

@torch.no_grad()
def get_logits(model, loader):
    model.eval(); out = []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        out.append(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")).cpu())
    return torch.cat(out)
print("metrics ok")

metrics ok


In [10]:
# ── Train one (model, seed) ─────────────────────────────────────────────────
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_one(model_key, model_name, seed):
    set_seed(seed); sd = f"{OUTPUT_DIR}/{model_key}_seed{seed}"; os.makedirs(sd, exist_ok=True)
    cfg = CONFIG
    # script-aware training data
    train_df = train_full
    if model_key == cfg.get("banglabert_script_key","banglabert_l") and cfg["banglabert_script_only"] and "script" in train_full.columns:
        train_df = train_full[train_full["script"] == "bangla"].reset_index(drop=True)
        print(f"  script-aware: BanglaBERT trains on {len(train_df):,} Bangla-script rows")
    tok = AutoTokenizer.from_pretrained(model_name); coll = Collator(tok)
    lkw = dict(collate_fn=coll, num_workers=cfg["num_workers"], pin_memory=torch.cuda.is_available())
    train_ds = DS(train_df, tok, cfg["max_length"], label_enc)
    if cfg["use_balanced_sampler"]:
        train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"],
                                  sampler=make_balanced_sampler(train_df, label_enc, cfg["sampler_alpha"]),
                                  shuffle=False, drop_last=True, **lkw)
    else:
        train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=True, **lkw)
    val_loader  = DataLoader(DS(val_df,  tok, cfg["max_length"], label_enc), batch_size=cfg["eval_batch_size"], shuffle=False, **lkw)
    test_loader = DataLoader(DS(test_df, tok, cfg["max_length"], label_enc), batch_size=cfg["eval_batch_size"], shuffle=False, **lkw)

    model = Encoder(model_name, NUM_CLASSES, cfg["dropout"], cfg["head_hidden_dim"], cfg["n_msd"]).to(device)
    cw = build_class_weights(train_df[LABEL_COL], label_enc, beta=cfg["class_weight_beta"],
                             per_class_mult=cfg.get("per_class_loss_multiplier", {}))
    focal = FocalLoss(cfg["focal_gamma"], cw, cfg["label_smoothing"])
    opt = torch.optim.AdamW(uniform_params(model, cfg["encoder_lr"], cfg["head_lr"], cfg["weight_decay"]))
    nsteps = math.ceil(len(train_loader)/cfg["grad_accum_steps"]) * cfg["epochs"]
    sch = get_linear_schedule_with_warmup(opt, int(nsteps*cfg["warmup_ratio"]), nsteps)
    scaler = torch.amp.GradScaler("cuda") if (cfg["use_fp16"] and device.type == "cuda") else None
    fgm = FGM(model, cfg["fgm_epsilon"]) if cfg["use_fgm"] else None
    ema = EMA(model, cfg["ema_decay"]) if cfg["use_ema"] else None

    best, noimp = -1.0, 0
    for ep in range(cfg["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True); run, t0 = 0.0, time.time()
        for step, b in enumerate(train_loader, 1):
            b = {k: v.to(device) for k, v in b.items()}
            y = b["label"]
            with torch.autocast(device_type=device.type, enabled=scaler is not None):
                lg1 = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                if cfg["use_rdrop"]:
                    lg2 = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                    loss = 0.5*(focal(lg1, y) + focal(lg2, y)) + cfg["rdrop_alpha"] * rdrop_kl(lg1, lg2)
                else:
                    loss = focal(lg1, y)
                loss = loss / cfg["grad_accum_steps"]
            (scaler.scale(loss) if scaler else loss).backward()
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type=device.type, enabled=scaler is not None):
                    lg_adv = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                    loss_adv = focal(lg_adv, y) / cfg["grad_accum_steps"]
                (scaler.scale(loss_adv) if scaler else loss_adv).backward()
                fgm.restore()
            if step % cfg["grad_accum_steps"] == 0 or step == len(train_loader):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                (scaler.step(opt), scaler.update()) if scaler else opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(model)
            run += loss.item() * cfg["grad_accum_steps"]
        # validate with EMA weights
        if ema is not None: ema.apply_shadow(model)
        vmac = evaluate(model, val_loader)["macro_f1"]
        flag = ""
        if vmac > best:
            best, noimp = vmac, 0; torch.save(model.state_dict(), f"{sd}/best_model.pt"); flag = " ✅BEST"
        else: noimp += 1
        if ema is not None: ema.restore(model)
        print(f"  Ep{ep+1:2}/{cfg['epochs']} loss={run/max(len(train_loader),1):.4f} "
              f"val_macroF1={vmac:.4f} {(time.time()-t0)/60:.1f}min{flag}")
        if noimp >= cfg["patience"]: print(f"  early stop ep{ep+1}"); break

    model.load_state_dict(torch.load(f"{sd}/best_model.pt", map_location=device, weights_only=True))
    test_metrics = evaluate(model, test_loader)         # PURE 20% test
    torch.save(get_logits(model, val_loader),  f"{sd}/val_logits.pt")
    torch.save(get_logits(model, test_loader), f"{sd}/test_logits.pt")
    res = {"model": model_key, "seed": seed, "best_val_macro_f1": round(best, 4), "test_metrics": test_metrics}
    json.dump(res, open(f"{sd}/results.json", "w"), indent=2)
    print(f"  → [20% TEST] macroF1={test_metrics['macro_f1']:.4f} wF1={test_metrics['weighted_f1']:.4f} "
          f"acc={test_metrics['accuracy']:.4f} mcc={test_metrics['mcc']:.4f}")
    return res
print("trainer ok")

trainer ok


In [ ]:
# ── Run all ─────────────────────────────────────────────────────────────────
all_results = []
for mk in RUN_MODELS:
    for s in RUN_SEEDS:
        sd = f"{OUTPUT_DIR}/{mk}_seed{s}"
        if not FORCE_RETRAIN and os.path.exists(f"{sd}/results.json"):
            print(f"⏭  {mk} seed={s} done"); all_results.append(json.load(open(f"{sd}/results.json"))); continue
        print(f"\n▶ {mk} seed={s}")
        try: all_results.append(train_one(mk, CONFIG["models"][mk], s))
        except Exception as e:
            import traceback; print(f"❌ {mk} seed={s}: {e}"); traceback.print_exc()
        torch.cuda.empty_cache()
print(f"\n🎉 {len(all_results)}/{len(RUN_MODELS)*len(RUN_SEEDS)} runs")


▶ banglishbert seed=42


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Ep 1/8 loss=0.4662 val_macroF1=0.7292 13.5min ✅BEST
  Ep 2/8 loss=0.2333 val_macroF1=0.7803 10.7min ✅BEST
  Ep 3/8 loss=0.1658 val_macroF1=0.7947 10.2min ✅BEST
  Ep 4/8 loss=0.1233 val_macroF1=0.7998 10.4min ✅BEST
  Ep 5/8 loss=0.0972 val_macroF1=0.8022 10.4min ✅BEST
  Ep 6/8 loss=0.0799 val_macroF1=0.8019 10.3min
  Ep 7/8 loss=0.0683 val_macroF1=0.8024 10.3min ✅BEST
  Ep 8/8 loss=0.0609 val_macroF1=0.8013 10.2min
  → [20% TEST] macroF1=0.8101 wF1=0.8203 acc=0.8202 mcc=0.7255

▶ banglishbert seed=123


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

  Ep 1/8 loss=0.4639 val_macroF1=0.7208 14.2min ✅BEST
  Ep 2/8 loss=0.2340 val_macroF1=0.7836 14.0min ✅BEST
  Ep 3/8 loss=0.1674 val_macroF1=0.7994 13.9min ✅BEST
  Ep 4/8 loss=0.1265 val_macroF1=0.8008 14.0min ✅BEST
  Ep 5/8 loss=0.0995 val_macroF1=0.8046 14.0min ✅BEST
  Ep 6/8 loss=0.0807 val_macroF1=0.8036 14.0min
  Ep 7/8 loss=0.0680 val_macroF1=0.8028 13.9min
  Ep 8/8 loss=0.0612 val_macroF1=0.8008 13.9min
  early stop ep8
  → [20% TEST] macroF1=0.8103 wF1=0.8199 acc=0.8202 mcc=0.7262

▶ banglabert_l seed=42
  script-aware: BanglaBERT trains on 39,826 Bangla-script rows


pytorch_model.bin:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert_large
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

In [ ]:
# ── Per-run summary (20% test) ──────────────────────────────────────────────
if all_results:
    rows = [{"model": r["model"], "seed": r["seed"], **r["test_metrics"]} for r in all_results]
    sdf = pd.DataFrame(rows)
    pd.set_option("display.width", 200)
    print(sdf.to_string(index=False))
    print("\n── mean ± std per model ──")
    agg = sdf.groupby("model")[["macro_f1","weighted_f1","accuracy","mcc","auroc"]].agg(["mean","std"]).round(4)
    print(agg.to_string())
    sdf.to_csv(f"{OUTPUT_DIR}/per_run_summary.csv", index=False)
    print(f"\n✅ saved {OUTPUT_DIR}/per_run_summary.csv")

---
**Outputs per run:** `best_model.pt`, `val_logits.pt`, `test_logits.pt`, `results.json`.
NB06 optimises ensemble weights on the val logits and reports the fused ensemble on the **20% test**.
Toggle any of `use_fgm / use_rdrop / use_ema / use_balanced_sampler` off to generate NB08 ablation rows.
